In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import matplotlib.pyplot as plt
import xgboost as xgb
from datetime import timedelta
import plotly.graph_objs as go

In [ ]:
df = pd.read_csv('Latest Data/Segregated/G003_Dataset_12_2024.csv')

In [ ]:
# df = pd.read_csv('G003_Dataset_12_2024.csv')

In [ ]:
df['elapsed_months'] = df['elapsed_months'].fillna(1)
df['elapsed_months'] -= 1
df = df[df['elapsed_months'] >= 0]

In [ ]:
df.loc[:, df.columns.str.startswith('MC_')].columns

In [ ]:
cl1 = df.loc[:, df.columns.str.startswith('MC_')].columns.tolist()

In [ ]:
df_g003 = df[df['game_id'] == 'G003']

In [ ]:
df_g003['cohort'] = pd.to_datetime(df_g003['cohort'], format='%d/%m/%Y')
df_g003['cohort_month'] = df_g003['cohort'].dt.month

In [ ]:
def calculate_current_month(row):
    cohort_start = row['cohort']
    elapsed_months = row['elapsed_months']
    current_month = (cohort_start + pd.DateOffset(months=int(elapsed_months))).month
    return current_month

In [ ]:
df_g003['current_month'] = df_g003.apply(calculate_current_month, axis=1)

In [ ]:
df_g003['cohort_month'] = df_g003['cohort_month'].astype('category')
df_g003['current_month'] = df_g003['current_month'].astype('category')

In [ ]:
df_g003.columns

In [ ]:
# feature_columns = [
#     'day_7_payer_retention', 'day_14_payer_retention', 'day_1_payer_retention','day_1_cumulative_ltv_cac',
#     'day_30_payer_retention', 'day_7_net_revenue_retention', 'day_14_net_revenue_retention', 
#     'day_30_net_revenue_retention', 'day_7_cumulative_ltv_cac',
#     'client_actual_spend', 'cohort_month', 'm0_ltv_cac'] + cl1

In [ ]:
feature_columns = [
            "day_1_payer_retention",
            "day_7_payer_retention",
            "day_14_payer_retention",
            "day_30_payer_retention",
            "day_7_net_revenue_retention",
            "day_14_net_revenue_retention",
            "day_30_net_revenue_retention",
            "client_actual_spend",
            "day_1_cumulative_ltv_cac",
            "day_7_cumulative_ltv_cac",
            "day_14_cumulative_ltv_cac",
            "day_30_cumulative_ltv_cac",
            "m0_ltv_cac",
            "MC_Apple",
            "MC_Applovin",
            "MC_Appnext",
            "MC_Facebook",
            "MC_Freecash",
            "MC_Google",
            "MC_Inboxdollars",
            "MC_Ironsource",
            "MC_Liftoff",
            "MC_Mintegral",
            "MC_Moloco",
            "MC_Others",
            "MC_Rewarded Ads",
            "MC_Snapchat",
            "MC_Tiktok",
            "MC_Unity",
            "MC_advgo42ltdcy235",
            "MC_bluestacks_int",
            "MC_connectedrx",
            "MC_dv360_int",
            "MC_elevate",
            "MC_inboxdollars",
            "MC_wynxmarkets",
            "MC_xiaomi",
            "cohort_month"
        ]

In [ ]:
for lag in range(1, 4):
    df_g003[f'ltv_lag_{lag}'] = df_g003.groupby('cohort')['transaction_period_ltv_cac'].shift(lag)


In [ ]:
from datetime import datetime
current_month_start = pd.Timestamp.today().normalize().replace(day=1)  # first day of current month as a Timestamp (fix C1)
dft = df_g003[
    pd.to_datetime(df_g003['transaction_period'], format='%d/%m/%Y', errors='coerce') < current_month_start
].copy()

In [ ]:
df_g003 = df_g003[df_g003['transaction_period_ltv_cac'].notnull()]

In [ ]:
dft = dft[dft['transaction_period_ltv_cac'].notnull()]

In [ ]:
dft_a = dft[dft['elapsed_months'] > 0]

In [ ]:

X = dft_a[feature_columns + ['ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3', 'elapsed_months']]
y = dft_a['transaction_period_ltv_cac']

In [ ]:
# Time-based split (fix C2): train on earlier transaction months, test on the most recent 20%
_tp = pd.to_datetime(dft_a.loc[X.index, 'transaction_period'], format='%d/%m/%Y', errors='coerce')
_order = _tp.sort_values(kind='stable').index
_cut = int(len(_order) * 0.8)
X_train, X_test = X.loc[_order[:_cut]], X.loc[_order[_cut:]]
y_train, y_test = y.loc[_order[:_cut]], y.loc[_order[_cut:]]

In [ ]:
y

In [ ]:
def augment_data(X_train, y_train, columns_to_modify, percentage_changes):
    augmented_X_train = X_train.copy()
    augmented_y_train = y_train.copy()
    
    for pct in percentage_changes:
        X_train_aug = X_train.copy()
        y_train_aug = y_train.copy()

        X_train_aug[columns_to_modify] = X_train[columns_to_modify] * (1 - pct / 100)
        y_train_aug = y_train * (1 - pct / 100)
        X_train_aug['client_actual_spend'] = X_train['client_actual_spend'] * (1 + pct / 100)
        
        augmented_X_train = pd.concat([augmented_X_train, X_train_aug], ignore_index=True)
        augmented_y_train = pd.concat([augmented_y_train, y_train_aug], ignore_index=True)
    
    return augmented_X_train, augmented_y_train

columns_to_modify = ['ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3']


percentage_changes = [1, 2, -1, -2]

X_train_augmented, y_train_augmented = augment_data(
    X_train=X_train,
    y_train=y_train,
    columns_to_modify=columns_to_modify,
    percentage_changes=percentage_changes
)

print("Augmented X_train shape:", X_train_augmented.shape)
print("Augmented y_train shape:", y_train_augmented.shape)


In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
import plotly.graph_objs as go
def plot_actual_vs_predicted(y_train, y_test, train_pred, test_pred):
    # Training set plot
    trace_train_actual = go.Scatter(
        x=np.arange(len(y_train)), y=y_train, mode='lines+markers',
        name='Actual (Train)', line=dict(color='blue'), marker=dict(size=5)
    )
    trace_train_pred = go.Scatter(
        x=np.arange(len(train_pred)), y=train_pred, mode='lines+markers',
        name='Predicted (Train)', line=dict(color='red', dash='dash'), marker=dict(size=5)
    )
    train_layout = go.Layout(
        title='Actual vs Predicted - Training Set',
        xaxis=dict(title='Index', tickmode='linear'),
        yaxis=dict(title='Values'),
        hovermode='x unified'
    )
    train_fig = go.Figure(data=[trace_train_actual, trace_train_pred], layout=train_layout)
    train_fig.show()

    # Test set plot
    trace_test_actual = go.Scatter(
        x=np.arange(len(y_test)), y=y_test, mode='lines+markers',
        name='Actual (Test)', line=dict(color='blue'), marker=dict(size=5)
    )
    trace_test_pred = go.Scatter(
        x=np.arange(len(test_pred)), y=test_pred, mode='lines+markers',
        name='Predicted (Test)', line=dict(color='red', dash='dash'), marker=dict(size=5)
    )
    test_layout = go.Layout(
        title='Actual vs Predicted - Test Set',
        xaxis=dict(title='Index', tickmode='linear'),
        yaxis=dict(title='Values'),
        hovermode='x unified'
    )
    test_fig = go.Figure(data=[trace_test_actual, trace_test_pred], layout=test_layout)
    test_fig.show()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from scipy.stats import uniform, randint

# Define custom quantile loss function
def quantile_loss(alpha):
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)  # Gradient
        hess = np.ones_like(y_true)  # Hessian
        return grad, hess
    return custom_loss

alpha = 0.5  # Change alpha to the desired quantile (e.g., 0.25, 0.5, 0.75)
model = xgb.XGBRegressor(objective=quantile_loss(alpha), enable_categorical=True)

# Preprocessing
numerical_columns = X_train.select_dtypes(include=['int64', 'float64']).columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
    ],
    remainder='passthrough'
)

# Pipeline: includes preprocessor and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Hyperparameter tuning
param_dist = {
    'model__n_estimators': randint(50, 500),
    'model__learning_rate': uniform(0.01, 0.09),
    'model__max_depth': randint(3, 10),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__min_child_weight': randint(1, 10),
    'model__gamma': uniform(0, 0.3),
    'model__reg_alpha': uniform(0, 1),
    'model__reg_lambda': uniform(1, 2),
}

# RandomizedSearchCV for hyperparameter tuning
random_search = RandomizedSearchCV(
    pipeline, param_distributions=param_dist,
    n_iter=100, scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
    cv=TimeSeriesSplit(n_splits=5), verbose=1, n_jobs=-1, random_state=42
)

random_search.fit(X_train_augmented, y_train_augmented)

best_model1 = random_search.best_estimator_


In [ ]:
from sklearn.metrics import mean_squared_error
y_train_pred = best_model1.predict(X_train)
y_test_pred = best_model1.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Train RMSE: {rmse_train}")
print(f"Test RMSE: {rmse_test}")

In [ ]:
plot_actual_vs_predicted(y_train, y_test, y_train_pred, y_test_pred)

In [ ]:
xgb_model = best_model1.named_steps['model']

encoded_feature_names = best_model1.named_steps['preprocessor'].get_feature_names_out()

feature_importances = xgb_model.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': encoded_feature_names,
    'Importance': feature_importances
})

feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
print(feature_importance_df)

In [ ]:
import cloudpickle

# Save the model with cloudpickle
with open('G003_xgb_0.5.pkl', 'wb') as f:
    cloudpickle.dump(best_model1, f)
print("Model saved successfully!")

25 percentile

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from scipy.stats import uniform, randint

# Define custom quantile loss function
def quantile_loss(alpha):
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)  # Gradient
        hess = np.ones_like(y_true)  # Hessian
        return grad, hess
    return custom_loss

alpha = 0.25  # Change alpha to the desired quantile (e.g., 0.25, 0.5, 0.75)
model = xgb.XGBRegressor(objective=quantile_loss(alpha), enable_categorical=True)

# Preprocessing
numerical_columns = X_train.select_dtypes(include=['int64', 'float64']).columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
    ],
    remainder='passthrough'
)

# Pipeline: includes preprocessor and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Hyperparameter tuning
param_dist = {
    'model__n_estimators': randint(50, 500),
    'model__learning_rate': uniform(0.01, 0.09),
    'model__max_depth': randint(3, 10),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__min_child_weight': randint(1, 10),
    'model__gamma': uniform(0, 0.3),
    'model__reg_alpha': uniform(0, 1),
    'model__reg_lambda': uniform(1, 2),
}

# RandomizedSearchCV for hyperparameter tuning
random_search = RandomizedSearchCV(
    pipeline, param_distributions=param_dist,
    n_iter=100, scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
    cv=TimeSeriesSplit(n_splits=5), verbose=1, n_jobs=-1, random_state=42
)

random_search.fit(X_train_augmented, y_train_augmented)

best_model2 = random_search.best_estimator_


In [ ]:
y_train_pred = best_model2.predict(X_train)
y_test_pred = best_model2.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Train RMSE: {rmse_train}")
print(f"Test RMSE: {rmse_test}")

In [ ]:
xgb_model = best_model2.named_steps['model']

encoded_feature_names = best_model2.named_steps['preprocessor'].get_feature_names_out()

feature_importances = xgb_model.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': encoded_feature_names,
    'Importance': feature_importances
})

feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
print(feature_importance_df)

In [ ]:
plot_actual_vs_predicted(y_train, y_test, y_train_pred, y_test_pred)

In [ ]:
with open('G003_xgb_0.25.pkl', 'wb') as f:
    cloudpickle.dump(best_model2, f)
print("Model saved successfully!")

75 percentile

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from scipy.stats import uniform, randint

# Define custom quantile loss function
def quantile_loss(alpha):
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)  # Gradient
        hess = np.ones_like(y_true)  # Hessian
        return grad, hess
    return custom_loss

alpha = 0.75  # Change alpha to the desired quantile (e.g., 0.25, 0.5, 0.75)
model = xgb.XGBRegressor(objective=quantile_loss(alpha), enable_categorical=True)

# Preprocessing
numerical_columns = X_train.select_dtypes(include=['int64', 'float64']).columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

param_dist = {
    'model__n_estimators': randint(50, 500),
    'model__learning_rate': uniform(0.01, 0.09),
    'model__max_depth': randint(3, 10),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__min_child_weight': randint(1, 10),
    'model__gamma': uniform(0, 0.3),
    'model__reg_alpha': uniform(0, 1),
    'model__reg_lambda': uniform(1, 2),
}

random_search = RandomizedSearchCV(
    pipeline, param_distributions=param_dist,
    n_iter=100, scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
    cv=TimeSeriesSplit(n_splits=5), verbose=1, n_jobs=-1, random_state=42
)

random_search.fit(X_train_augmented, y_train_augmented)

best_model3 = random_search.best_estimator_


In [ ]:
y_train_pred = best_model3.predict(X_train)
y_test_pred = best_model3.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Train RMSE: {rmse_train}")
print(f"Test RMSE: {rmse_test}")

In [ ]:
plot_actual_vs_predicted(y_train, y_test, y_train_pred, y_test_pred)

In [ ]:
with open('G003_xgb_0.75.pkl', 'wb') as f:
    cloudpickle.dump(best_model3, f)
print("Model saved successfully!")

In [ ]:
sp_ltv_cac= df[df['transaction_period_ltv_cac'] > 0]['transaction_period_ltv_cac'].min()

In [ ]:
def forecast_cumulative_ltv_cac_variations(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12):
    game_data = df[df['game_id'] == game_id].copy()
    forecasted_results = []

    for cohort in game_data['cohort'].unique():
        cohort_data = game_data[game_data['cohort'] == cohort].copy()
        m0_ltv_cac = cohort_data['m0_ltv_cac'].iloc[0]
        total_cac = max(cohort_data['client_actual_spend'].iloc[0],cohort_data['total_cac'].iloc[0])
        last_elapsed_month = int(cohort_data['elapsed_months'].max())
        cumulative_actual = m0_ltv_cac

        # Calculate cumulative actual values until the last known elapsed month
        actual_cumulative = []
        for month in range(1, last_elapsed_month + 1):
            transactional_actual = cohort_data.loc[cohort_data['elapsed_months'] == month, 'transaction_period_ltv_cac'].values
            transactional_actual = transactional_actual[0] if len(transactional_actual) > 0 else 0
            cumulative_actual += transactional_actual
            actual_cumulative.append({
                'month': month,
                'transactional_actual': transactional_actual,
                'cumulative_actual': cumulative_actual,
                'absolute_revenue_actual': total_cac * transactional_actual,
                'total_cac': total_cac
            })

        lag_1 = np.nan
        lag_2 = np.nan
        lag_3 = np.nan
        lag_1b = np.nan
        lag_1g = np.nan

        cumulative_values = {key: m0_ltv_cac for key in ['cumulative_bad', 'cumulative_expected', 'cumulative_good']}

        # Use static features from any month (e.g., last available month)
        static_features = cohort_data.loc[cohort_data['elapsed_months'] == last_elapsed_month, best_model_expected.feature_names_in_].iloc[0].copy()

        # Remove features that need to change: 'elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3'
        for col in ['elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3']:
            if col in static_features:
                static_features = static_features.drop(col)

        # Predict for each month up to the last actual month + forecast period
        for i in range(1, last_elapsed_month + forecast_months + 1):
            elapsed_months = i

            # Prepare features
            features = static_features.copy()

            # Update 'elapsed_months'
            features['elapsed_months'] = elapsed_months

            # Update 'current_month' if it is a feature
            if 'current_month' in best_model_expected.feature_names_in_:
                # Calculate 'current_month' based on cohort start date
                cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])
                current_date = cohort_start_date + pd.DateOffset(months=elapsed_months)
                features['current_month'] = current_date.month  # Month as integer from 1 to 12
            else:
                current_date = pd.to_datetime(cohort_data['cohort'].iloc[0]) + pd.DateOffset(months=elapsed_months)

            # Update lag features
            features1 = features.copy()
            features2 = features.copy()


            features['ltv_lag_1'] = lag_1
            features1['ltv_lag_1'] = lag_1b
            features2['ltv_lag_1'] = lag_1g
            features['ltv_lag_2'] = lag_2
            features1['ltv_lag_2'] = lag_2
            features2['ltv_lag_2'] = lag_2
            features['ltv_lag_3'] = lag_3
            features1['ltv_lag_3'] = lag_3
            features2['ltv_lag_3'] = lag_3

            # Ensure correct format for model prediction
            features = pd.DataFrame([features])
            features1 = pd.DataFrame([features1])
            features2 = pd.DataFrame([features2])

            # Predictions for each model
            predicted_val_bad = max(0, best_model_bad.predict(features1)[0])  
            predicted_val_expected = max(0, best_model_expected.predict(features)[0])
            predicted_val_good = max(0, best_model_good.predict(features2)[0])
            x = [predicted_val_bad,predicted_val_expected,predicted_val_good]
            x.sort()
            predicted_val_bad = x[0]
            predicted_val_expected = x[1] 
            predicted_val_good = x[2]


            if(predicted_val_good<=0):
                predicted_val_good=sp_ltv_cac
            if(predicted_val_expected<=0):
                predicted_val_expected = predicted_val_good/2
            if(predicted_val_bad<=0):
                predicted_val_bad = predicted_val_expected/2


            # Update lag features for next month
            lag_3 = lag_2
            lag_2 = lag_1
            lag_1b = predicted_val_bad if i > last_elapsed_month else cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac'].values[0]
            lag_1g = predicted_val_good if i > last_elapsed_month else cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac'].values[0]
            lag_1 = predicted_val_expected if i > last_elapsed_month else cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac'].values[0]

            # Prepare result row
            row_result = {
                'game_id': game_id,
                'cohort': cohort,
                'elapsed_months': elapsed_months,
                'current_month': current_date.strftime('%Y-%m'),
            }

            # Add actuals if within actual data range
            actual_data = next((ac for ac in actual_cumulative if ac['month'] == elapsed_months), None)
            row_result['transactional_actual'] = actual_data['transactional_actual'] if actual_data else None
            row_result['cumulative_actual'] = actual_data['cumulative_actual'] if actual_data else None
            row_result['absolute_revenue_actual'] = actual_data['absolute_revenue_actual'] if actual_data else None
            row_result['total_cac'] = actual_data['total_cac'] if actual_data else total_cac

            # Store predictions and cumulative values
            for model_type, predicted_val in zip(
                ['bad', 'expected', 'good'],
                [predicted_val_bad, predicted_val_expected, predicted_val_good]
            ):
                cumulative_key = f"cumulative_{model_type}"
                cumulative_values[cumulative_key] += predicted_val
                row_result[f"transactional_{model_type}"] = predicted_val
                row_result[cumulative_key] = cumulative_values[cumulative_key]
                row_result[f"absolute_revenue_{model_type}"] = total_cac * predicted_val

            forecasted_results.append(row_result)

    forecast_df = pd.DataFrame(forecasted_results)
    return forecast_df


In [ ]:
df1 = forecast_cumulative_ltv_cac_variations('G003', best_model1,best_model2,best_model3, df_g003, 12)

In [ ]:
df1.to_csv('G003_predictions.csv',index=False)

In [ ]:
def plot_ltv_cac_forecasts_for_game(df, game_id, end_month):
    # Filter data for the specific game_id
    game_data = df[df['game_id'] == game_id].copy()

    # Loop through each cohort within the game_id data
    for cohort in game_data['cohort'].unique():
        print(f"Generating plots for Game ID: {game_id}, Cohort: {cohort}")
        plot_ltv_cac_forecasts(df, cohort, end_month)

def plot_ltv_cac_forecasts(df, cohort, end_month):
    # Filter data for the specific cohort
    cohort_data = df[df['cohort'] == cohort].copy()
    cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])

    # Prepare data for plotting
    actual_transactional = cohort_data['transactional_actual']
    actual_cumulative = cohort_data['cumulative_actual']
    
    predicted_transactional_bad = cohort_data['transactional_bad']
    predicted_transactional_expected = cohort_data['transactional_expected']
    predicted_transactional_good = cohort_data['transactional_good']
    
    predicted_cumulative_bad = cohort_data['cumulative_bad']
    predicted_cumulative_expected = cohort_data['cumulative_expected']
    predicted_cumulative_good = cohort_data['cumulative_good']

    months = cohort_data['current_month']

    # Plotting Transactional LTV CAC
    trace_actual_transactional = go.Scatter(
        x=months,
        y=actual_transactional,
        mode='lines+markers',
        name='Actual Transactional LTV CAC',
        line=dict(color='blue', width=2),
        marker=dict(size=6, symbol='circle')
    )
    trace_predicted_transactional_bad = go.Scatter(
        x=months,
        y=predicted_transactional_bad,
        mode='lines+markers',
        name='Predicted Transactional LTV CAC (Bad)',
        line=dict(color='red', dash='dot', width=2),
        marker=dict(size=6, symbol='diamond')
    )
    trace_predicted_transactional_expected = go.Scatter(
        x=months,
        y=predicted_transactional_expected,
        mode='lines+markers',
        name='Predicted Transactional LTV CAC (Expected)',
        line=dict(color='orange', dash='dot', width=2),
        marker=dict(size=6, symbol='diamond')
    )
    trace_predicted_transactional_good = go.Scatter(
        x=months,
        y=predicted_transactional_good,
        mode='lines+markers',
        name='Predicted Transactional LTV CAC (Good)',
        line=dict(color='green', dash='dot', width=2),
        marker=dict(size=6, symbol='diamond')
    )

    layout_transactional = go.Layout(
        title=f'Transactional LTV CAC (Actual vs Predicted) for Cohort {cohort}',
        xaxis=dict(
            title='Month',
            tickmode='array',
            tickvals=np.arange(1, end_month + 1),
            ticktext=months
        ),
        yaxis=dict(title='LTV CAC'),
        hovermode='x unified'
    )

    fig_transactional = go.Figure(data=[
        trace_actual_transactional,
        trace_predicted_transactional_bad,
        trace_predicted_transactional_expected,
        trace_predicted_transactional_good
    ], layout=layout_transactional)
    fig_transactional.show()

    # Plotting Cumulative LTV CAC
    trace_actual_cumulative = go.Scatter(
        x=months,
        y=actual_cumulative,
        mode='lines+markers',
        name='Actual Cumulative LTV CAC',
        line=dict(color='blue', width=2),
        marker=dict(size=6, symbol='circle')
    )
    trace_predicted_cumulative_bad = go.Scatter(
        x=months,
        y=predicted_cumulative_bad,
        mode='lines+markers',
        name='Predicted Cumulative LTV CAC (Bad)',
        line=dict(color='red', dash='dot', width=2),
        marker=dict(size=6, symbol='diamond')
    )
    trace_predicted_cumulative_expected = go.Scatter(
        x=months,
        y=predicted_cumulative_expected,
        mode='lines+markers',
        name='Predicted Cumulative LTV CAC (Expected)',
        line=dict(color='orange', dash='dot', width=2),
        marker=dict(size=6, symbol='diamond')
    )
    trace_predicted_cumulative_good = go.Scatter(
        x=months,
        y=predicted_cumulative_good,
        mode='lines+markers',
        name='Predicted Cumulative LTV CAC (Good)',
        line=dict(color='green', dash='dot', width=2),
        marker=dict(size=6, symbol='diamond')
    )

    layout_cumulative = go.Layout(
        title=f'Cumulative LTV CAC (Actual vs Predicted) for Cohort {cohort}',
        xaxis=dict(
            title='Month',
            tickmode='array',
            tickvals=np.arange(1, end_month + 1),
            ticktext=months
        ),
        yaxis=dict(title='Cumulative LTV CAC'),
        hovermode='x unified'
    )

    fig_cumulative = go.Figure(data=[
        trace_actual_cumulative,
        trace_predicted_cumulative_bad,
        trace_predicted_cumulative_expected,
        trace_predicted_cumulative_good
    ], layout=layout_cumulative)
    fig_cumulative.show()



In [ ]:
plot_ltv_cac_forecasts_for_game(df1, game_id='G003', end_month=12)